# Topic 07 — Practice: DateTime & Time Series

**The notebook is the lesson.** Work top to bottom. Cells with `assert` grade themselves — a silent run = pass, an `AssertionError` = fix your code. Warm-Up, Interview Lens and Reflection have **no answer key** — answer in writing.

_Spaced repetition: ~60% of this notebook is the current topic, ~40% revisits earlier topics._

In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 30)
RAW = '../datasets/raw/'
orders = pd.read_csv(RAW+'orders.csv', dtype={'order_id':str})
orders['channel'] = orders['channel'].str.strip().str.title()
items = pd.read_csv(RAW+'order_items.csv', dtype={'order_id':str,'product_id':str})
items['line_revenue'] = items['quantity']*items['unit_price']*(1-items['discount'])


## 🔁 Warm-Up Recall (earlier topics — no answers given)

From **Topics 01–06**:

1. (T06) Why does a left join sometimes increase the row count?
2. (T05) Write revenue-per-channel with named aggregation.
3. NumPy: what dtype underlies pandas datetimes?

In [ ]:
# NumPy recall: timedelta arithmetic
d = np.array(['2023-01-01','2023-02-01'], dtype='datetime64[D]')
gap = ...   # TODO days between the two (int)
assert gap == 31
print(gap)

## 🎯 Core Lesson Tasks (current topic)

Parse messy dates with format='mixed', dayfirst=True, errors='coerce'. Then `.dt`, `resample`, `rolling`.

**Core 1 — parse mixed dates.** Parse `orders['order_date']` robustly; report the parsed fraction.

In [ ]:
orders['order_date'] = ...
assert orders['order_date'].notna().mean() > 0.95
print(orders['order_date'].notna().mean())

**Core 2 — monthly revenue.** Join line revenue to dates, set a datetime index, resample to monthly sum.

In [ ]:
oi = items.merge(orders[['order_id','order_date']], on='order_id', how='left')
monthly = ...
assert monthly.index.is_monotonic_increasing
monthly.tail()

**Core 3 — growth.** Month-over-month growth via `pct_change`; print the worst and best month.

In [ ]:
mom = ...
assert mom.notna().sum() > 10
print('worst', mom.idxmin(), '| best', mom.idxmax())

## 🔀 Mixed Review Tasks (earlier topics — spaced repetition)

**Mixed review (Topic 05) — group by time part.** Total revenue by day-of-week name (use `.dt.day_name()`).

In [ ]:
oi['dow'] = oi['order_date'].dt.day_name()
dow_rev = ...
assert len(dow_rev) <= 7
dow_rev

**Mixed review (Topic 04) — gaps.** `marketing_spend` is missing a few days. Reindex daily spend to a gap-free range; count filled days.

In [ ]:
mk = pd.read_csv(RAW+'marketing_spend.csv', parse_dates=['date'])
daily = mk.groupby('date')['spend_gbp'].sum()
full = pd.date_range(daily.index.min(), daily.index.max(), freq='D')
n_missing = ...
assert n_missing >= 0
print('missing days:', n_missing)

## 🕵️ Data Detective Investigation

**Case file #7 — is the dip real?** The finance team panicked over a revenue 'dip'. Plot monthly revenue with a 3-month moving average. Once the dates are parsed correctly, decide: real fall, or an artifact of dirty dates / seasonality? Write your verdict in the reflection.

In [ ]:
import matplotlib.pyplot as plt
oi = items.merge(orders[['order_id','order_date']], on='order_id', how='left')
ts = oi.dropna(subset=['order_date']).set_index('order_date').sort_index()
monthly = ts['line_revenue'].resample('ME').sum()
ax = monthly.plot(figsize=(10,4)); monthly.rolling(3).mean().plot(ax=ax)
ax.set_title('Verdict: real dip or seasonal artifact?'); ax.set_ylabel('Revenue £'); plt.show()
assert len(monthly) >= 12
print('months:', len(monthly))

## 🔎 Interview Lens (write answers — no model answers)

- **Q22:** Why is parsing at load better than converting after? When is the reverse true?
- **Q23:** `resample` vs `groupby` on a datetime — when interchangeable?
- **Q24:** Handling gaps in a daily series — `fillna` vs interpolation risks?

## ✍️ Reflection (write short explanations)

1. Answer Q22 and Q24 in writing.
2. Is the dip real? State your evidence.
3. **Investigation log:** what does the time pattern say about Aurora's revenue?